# Data Definition and Schema Design (DDL)

Now that we understand the theory of how tables relate to one another, it is time to actually build them. In SQL, the commands used to create, modify, and delete the structure of a database are called **Data Definition Language (DDL)**. 

Think of DDL as the architectural blueprint of your database. You aren't adding the actual data (the furniture) yet; you are just building the rooms, the walls, and deciding what type of things are allowed inside each room.

Since we are Data Scientists, we will use Python's built-in `sqlite3` library to write and execute real SQL code right here in the notebook. This creates an "in-memory" database—a temporary sandbox perfect for learning.

---

# 1. Understanding Data Types
When you create a column in a table, you must tell the database exactly what *type* of data goes there. This prevents bad data from ruining your analysis later (like someone typing "Twenty" instead of `20` in an age column).

Common SQL Data Types:
* **`INT` / `INTEGER`**: Whole numbers (e.g., `1`, `42`, `-5`).
* **`VARCHAR(n)` / `TEXT`**: Text or strings. `VARCHAR(50)` means text up to 50 characters long.
* **`DECIMAL(p, s)` / `FLOAT`**: Numbers with decimals (e.g., `19.99`).
* **`DATE` / `TIMESTAMP`**: Dates and exact times (e.g., `2024-05-12 08:30:00`).
* **`BOOLEAN`**: True or False (often stored as `1` or `0` in some databases).

# 2. Creating Tables (`CREATE TABLE`)
To create a table, you define the table name, column names, data types, and any constraints (like `PRIMARY KEY` or `NOT NULL`).

Let's use Python to create our very first database table

In [1]:
import sqlite3
import pandas as pd

# 1. Create a temporary "in-memory" database (disappears when the notebook closes)
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

# 2. Write the SQL DDL command to create a table
create_table_sql = """
CREATE TABLE Employees (
    employee_id INTEGER PRIMARY KEY,
    first_name TEXT NOT NULL,
    last_name TEXT NOT NULL,
    hire_date DATE,
    salary DECIMAL(10, 2)
);
"""

# 3. Execute the SQL command
cursor.execute(create_table_sql)
print("✅ Table 'Employees' created successfully!")

✅ Table 'Employees' created successfully!


# 3. Viewing the Schema
How do we know it worked? We can ask the database to show us its "schema" (the structural blueprint of the table).


In [2]:
# Query the schema information for the 'Employees' table in SQLite
schema_query = "PRAGMA table_info(Employees);"
schema_data = pd.read_sql_query(schema_query, conn)

print("--- Employees Table Blueprint ---")
display(schema_data[['name', 'type', 'notnull', 'pk']]) 
# 'notnull' shows 1 if it's required, 'pk' shows 1 if it's the Primary Key

--- Employees Table Blueprint ---


,name,type,notnull,pk
0,employee_id,INTEGER,0,1
1,first_name,TEXT,1,0
2,last_name,TEXT,1,0
3,hire_date,DATE,0,0
4,salary,"DECIMAL(10, 2)",0,0


# 4. Modifying Tables (`ALTER TABLE`)
Business requirements change all the time. Imagine your boss says, "We need to start tracking employee phone numbers." You don't have to delete the table and start over; you can alter it.

In [3]:
# Add a new column to the existing table
alter_table_sql = """
ALTER TABLE Employees
ADD COLUMN phone_number TEXT;
"""
cursor.execute(alter_table_sql)

# Let's check the schema again to see the new column
updated_schema = pd.read_sql_query(schema_query, conn)
print("\n✅ Table 'Employees' updated! New Blueprint:")
display(updated_schema[['name', 'type']])


✅ Table 'Employees' updated! New Blueprint:


,name,type
0,employee_id,INTEGER
1,first_name,TEXT
2,last_name,TEXT
3,hire_date,DATE
4,salary,"DECIMAL(10, 2)"
5,phone_number,TEXT


*(Note: Different databases like PostgreSQL or MySQL allow you to DROP or RENAME columns easily with `ALTER TABLE`, but SQLite has stricter limits on altering tables. Adding columns is universally supported!)*

# 5. Deleting Tables (`DROP` vs `TRUNCATE`)
If you make a massive mistake, or an old table is no longer needed, you can delete it. Be incredibly careful with these commands in the real world!

* **`DROP TABLE table_name;`**: Completely destroys the table, its structure, and all data inside it. The blueprint is burned.
* **`TRUNCATE TABLE table_name;`**: Deletes all the *data* (rows) inside the table, but leaves the empty structure intact. *(Note: SQLite uses `DELETE FROM table_name;` for this purpose instead).*

In [4]:
# Let's destroy our table
drop_table_sql = "DROP TABLE Employees;"
cursor.execute(drop_table_sql)
print("🗑️ Table 'Employees' has been completely deleted.")

# Close our database connection
conn.close()

🗑️ Table 'Employees' has been completely deleted.


## Real-World Use Case or Analogy:
Think of Data Definition Language (DDL) like **Building a House**:

* **`CREATE TABLE`**: You are pouring the foundation and framing the walls. You decide that the kitchen (`VARCHAR`) will be here, and the bathroom (`INTEGER`) will be there. You haven't moved your furniture in yet; you are just establishing the structure.
* **Data Types & Constraints**: These are the building codes. You put a lock on the front door (`PRIMARY KEY`) and ensure the bathroom has plumbing (`NOT NULL`). If someone tries to put a car in the living room (wrong data type), the house rejects it.
* **`ALTER TABLE`**: You've lived in the house for a year, and now you want to add a sunroom. You hire contractors to build an extension without tearing the rest of the house down.
* **`TRUNCATE TABLE`**: You hire movers to take absolutely all your furniture and belongings out of the house. The empty house still stands.
* **`DROP TABLE`**: You hire a demolition crew with a wrecking ball. The house, the furniture, and the foundation are entirely wiped off the map.

---